# 03d - Diagnostico de outliers (grao anual)

Diagnostico por variavel: regra IQR (limite superior Q3 + 1.5*IQR), percentis extremos
(p99, p99.9, max), share acima do limite IQR e razao max/p99. Distingue **cauda
estrutural** (manter, tratar com log/rank/robustez) de **anomalia de dado**. Nenhum
outlier e tratado automaticamente como erro.

In [1]:
import numpy as np, pandas as pd
from pathlib import Path
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PROCESSED = PROJECT_ROOT/"data"/"processed"; TABLES = PROJECT_ROOT/"reports"/"tables"
TARGET = "custo_ano_real"
df = pd.read_csv(DATA_PROCESSED/"base_anual_carreta_deflacionada.csv")
VARS = [TARGET] + [c for c in ["idade_carreta","km_acumulado_fim_ano","km_rodado_ano",
        "n_os_ano","custo_medio_por_os_ano","n_sistemas_vmrs_distintos_ano","share_pm_ano",
        "n_os_ano_anterior","n_os_acum_ate_ano_anterior","custo_ano_anterior",
        "custo_acum_ate_ano_anterior","ano_modelo","eixos","comprimento"] if c in df.columns]
rows = []
for c in VARS:
    v = pd.to_numeric(df[c], errors="coerce").dropna()
    q1, q3 = v.quantile(.25), v.quantile(.75); iqr = q3 - q1
    lim = q3 + 1.5*iqr
    p99, p999, mx = v.quantile(.99), v.quantile(.999), v.max()
    share = float((v > lim).mean()); razao = (mx/p99) if p99 else np.nan
    if c == TARGET:
        dec = "cauda estrutural; log1p/robustez na modelagem; negativos ja excluidos"
    elif razao and razao > 20:
        dec = "winsorizar/monitorar (cauda muito longa)"
    elif share > 0.10:
        dec = "cauda longa estrutural (manter; log/rank)"
    elif share > 0:
        dec = "outliers moderados (manter)"
    else:
        dec = "sem outliers relevantes"
    rows.append({"variavel": c, "Q1": round(q1,1), "Q3": round(q3,1), "lim_sup_iqr": round(lim,1),
                 "p99": round(p99,1), "p99_9": round(p999,1), "max": round(mx,1),
                 "share_acima_iqr": round(share,4), "razao_max_p99": round(razao,1) if razao==razao else None,
                 "decisao": dec})
out = pd.DataFrame(rows)
out.to_csv(TABLES/"03d_diagnostico_outliers.csv", index=False)
print(out.to_string(index=False))

                     variavel      Q1       Q3  lim_sup_iqr      p99     p99_9       max  share_acima_iqr  razao_max_p99                                                               decisao
               custo_ano_real   317.8   2009.9       4548.1  11804.4   20150.9   62230.9           0.0886            5.3 cauda estrutural; log1p/robustez na modelagem; negativos ja excluidos
                idade_carreta     4.0     12.0         24.0     23.0      26.0      39.0           0.0040            1.7                                           outliers moderados (manter)
         km_acumulado_fim_ano 39267.5 238422.8     537155.6 878567.9 1308892.4 3479389.0           0.0632            4.0                                           outliers moderados (manter)
                km_rodado_ano  4591.0  34785.5      80077.2 133830.8  215890.6  249676.0           0.0539            1.9                                           outliers moderados (manter)
                     n_os_ano     2.0      6.